# 03 — Chatbot RAG Interactivo

MVP completo del chatbot. Requiere que `02_indexacion_vectorstore.ipynb` haya sido ejecutado (ChromaDB en `../chroma_db/`) y que Ollama esté corriendo con el modelo descargado.

**Prerequisito:** `ollama pull mistral:7b-instruct`

In [1]:
import sys
sys.path.insert(0, '..')

from src.rag_chain import construir_cadena, formatear_respuesta

# Cambiar el modelo aquí si se quiere probar otro
MODELO = 'mistral:7b-instruct'  # o 'llama3.1:8b', 'gemma2:9b'

print(f'Cargando cadena RAG con modelo: {MODELO}')
chain = construir_cadena(model=MODELO, k=3, chroma_dir='../chroma_db')
print('Listo.')

Cargando cadena RAG con modelo: mistral:7b-instruct


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Listo.


In [2]:
def preguntar(pregunta: str, mostrar_fuentes: bool = True):
    print(f'Pregunta: {pregunta}')
    print('-' * 60)
    resultado = chain(pregunta)
    print(resultado['result'])
    if mostrar_fuentes:
        print()
        fuentes = {(d.metadata['nombre_doc'], d.metadata['seccion'])
                   for d in resultado.get('source_documents', [])}
        print('Fuentes consultadas:')
        for nombre, seccion in sorted(fuentes):
            print(f'  • {nombre} — {seccion[:70]}')
    print()

In [3]:
# Prueba 1: Producto defectuoso
preguntar('¿Cuáles son mis derechos si un producto que compré está defectuoso?')

Pregunta: ¿Cuáles son mis derechos si un producto que compré está defectuoso?
------------------------------------------------------------
 Si compraste un producto y este tiene un defecto, tienes derecho a reclamar su reparación o reposición. El tipo de defecto puede ser por deficiencias en la fabricación, estructura, calidad, condiciones sanitarias, materiales utilizados, etc., o incluso que el producto no sea apto para el uso para el cual está destinado.

Si se trata de un problema con la medición del producto, tienes derecho a reclamar dentro de los diez (10) días hábiles siguientes a la fecha en que se advierta la deficiencia de la medición o el instrumento utilizado para ella.

Si lo compré con una garantía legal o voluntaria, tienes derecho a devolver el producto y recibir tu dinero de vuelta, o bien, a que se repara o reemplace el mismo. Si la devolución es la opción elegida, el valor del producto en el momento de la devolución será la base para determinar el monto a ser restit

In [4]:
# Prueba 2: Reclamo ante INDECOPI
preguntar('¿Cómo presento una queja ante INDECOPI?')

Pregunta: ¿Cómo presento una queja ante INDECOPI?
------------------------------------------------------------
 Para presentar una queja ante INDECOPI (Instituto Nacional de Defensa de la Competencia y Protección de la Propiedad Intelectual), sigue estos pasos:

1. Identificate a ti mismo proporcionando tu nombre completo, domicilio de notificación, número de DNI, Carné de Extranjería (C.E.) o Pasaporte.
2. Especifica la empresa o servicio quejado.
3. Detalla tu queja.
4. Manifesta tu voluntad para ser contactado por correo electrónico si es necesario.
5. Autoriza el acceso a tus datos médicos en caso de que sea relevante (por ejemplo, en casos relacionados con productos o servicios sanitarios).
6. Firma tu queja en físico (en caso de presentarse personalmente) o regularizar la consignación de la firma durante el procedimiento de atención de la queja (si presentas tu queja electrónicamente o verbalmente). Si tienes dificultades para firmar, solo imprimirás tu huella digital.
7. En caso

In [5]:
# Prueba 3: SOAT
preguntar('¿Qué cubre el SOAT en caso de accidente de tránsito?')

Pregunta: ¿Qué cubre el SOAT en caso de accidente de tránsito?
------------------------------------------------------------
 El Seguro Obligatorio contra Accidentes de Tránsito (SOAT) cubre a la persona que está involucrada en un accidente de tránsito por daños materiales o personales que pudieran resultar de dicho suceso. No importa si el vehículo asegurado sea el responsable del accidente o no, el SOAT le brindará una cobertura en caso de ser víctima de un accidente de tránsito. El monto de la indemnización por muerte es fijado legalmente y no se pueden otorgar dos veces esta misma indemnización dentro del marco de este tipo de seguro.

Fuente: Ley N° 26735, Artículo 18; Reglamento del SOAT, Artículos 9 y 10.

Fuentes consultadas:
  • Lineamientos sobre Protección al Consumidor - Actualizado 2022 — 12.1. Accidente de tránsito
  • Lineamientos sobre Protección al Consumidor - Actualizado 2022 — 12.2. Naturaleza del SOAT-CAT



In [6]:
# Prueba 4: Atención preferente
preguntar('¿Tengo derecho a atención preferente si soy adulto mayor?')

Pregunta: ¿Tengo derecho a atención preferente si soy adulto mayor?
------------------------------------------------------------
 Si eres un adulto mayor, sí tienes derecho a trato preferencial según la ley peruana. Eso significa que los proveedores deben atenderte antes que al resto de clientes en casos de igualdad. En caso de que se agote el módulo de atención preferencial, tendrás prioridad si te encuentras primero en la fila.

Fuentes consultadas:
  • Lineamientos sobre Protección al Consumidor - Actualizado 2022 — 2.3. Orden de atención



In [7]:
# Evaluación con ROUGE-L (comparar contra respuesta de referencia manual)
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

# Definir pares (pregunta, respuesta de referencia simplificada manualmente)
pares_evaluacion = [
    (
        '¿Qué es el libro de reclamaciones?',
        'El libro de reclamaciones es un registro donde puedes dejar constancia de '
        'tu queja o reclamo sobre un producto o servicio. Todo establecimiento que '
        'vende bienes o presta servicios está obligado a tenerlo y a dártelo cuando lo pidas.',
    ),
    (
        '¿Cuándo vence la garantía de un producto?',
        'La garantía legal de un producto es de 2 años desde que lo compraste. '
        'En ese plazo, si el producto tiene defectos de fabricación, el vendedor '
        'debe repararlo, cambiarlo o devolverte el dinero.',
    ),
]

print('Evaluación ROUGE-L (0 a 1, mayor es mejor)\n')
for pregunta, referencia in pares_evaluacion:
    resultado = chain(pregunta)
    respuesta = resultado['result']
    scores = scorer.score(referencia, respuesta)
    print(f'Pregunta: {pregunta}')
    print(f'ROUGE-L F1: {scores["rougeL"].fmeasure:.3f}')
    print()

Evaluación ROUGE-L (0 a 1, mayor es mejor)

Pregunta: ¿Qué es el libro de reclamaciones?
ROUGE-L F1: 0.224

Pregunta: ¿Cuándo vence la garantía de un producto?
ROUGE-L F1: 0.184



In [8]:
# Chat interactivo en el notebook
print('Chatbot activado. Escribe \'salir\' para terminar.\n')
while True:
    pregunta = input('Tú: ').strip()
    if pregunta.lower() in ('salir', 'exit', 'quit', ''):
        break
    resultado = chain(pregunta)
    print(f'\nAsistente: {resultado["result"]}\n')

Chatbot activado. Escribe 'salir' para terminar.


Asistente:  Hola! Soy un asistente que te ayuda a entender tus derechos como consumidor peruano. No respondo esta vez porque tu pregunta no está relacionada con tus derechos de consumidor. ¡Puedes preguntar sobre cualquier derecho de consumo y recibirás una respuesta clara, práctica y en lenguaje simple!

Fuente: Ninguna (respuesta generada por el sistema)

